# 03 — Integração e limpeza

**Objetivo:** juntar em uma única tabela, por município de SP:
- idosos sozinhos / total de idosos (notebook 01)
- internações por causa e ano (notebook 02)
- IDH municipal (Atlas Brasil — download manual)
- (opcional) cobertura de Estratégia Saúde da Família (download manual)

**Ponto de atenção — códigos de município:** o IBGE usa código de 7 dígitos,
o DATASUS usa 6 dígitos (sem o dígito verificador). Já geramos a tabela
`municipios_sp.csv` no notebook 01 com as duas versões — é ela que usamos
como "tradutor" entre as bases.

**Saída desta etapa:** `data/processed/dataset_consolidado_sp.csv`, com uma
linha por (município, ano) — a base que os notebooks 04 e 05 vão analisar.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd


## 3.1 Carregar as bases já processadas


In [ ]:
municipios = pd.read_csv(config.DATA_PROCESSED / "municipios_sp.csv")
censo = pd.read_csv(config.DATA_PROCESSED / "censo_domicilios_sp.csv")
internacoes = pd.read_csv(config.DATA_PROCESSED / "internacoes_sp.csv")

print("municipios:", municipios.shape)
print("censo:", censo.shape)
print("internacoes:", internacoes.shape)


## 3.2 IDH municipal (Atlas Brasil — download manual)

1. Acesse: http://www.atlasbrasil.org.br/ranking
2. Filtre Estado = São Paulo
3. Baixe em Excel e salve como `data/external/idh_sp.xlsx`

Essa variável entra como **controle**: o objetivo é checar se a associação
entre "idosos sozinhos" e "internações evitáveis" se mantém mesmo depois de
levar em conta a renda/desenvolvimento do município — sem isso, um
resultado poderia ser só reflexo de municípios mais pobres terem, ao mesmo
tempo, mais idosos sozinhos e piores indicadores de saúde por outros
motivos.


In [ ]:
caminho_idh = config.DATA_EXTERNAL / "idh_sp.xlsx"

if caminho_idh.exists():
    idh = pd.read_excel(caminho_idh)
    print(idh.columns.tolist())
    idh.head()
else:
    print(f"Baixe o arquivo conforme instruções acima e salve em {caminho_idh}")
    idh = None


**Depois de rodar a célula acima**, me diz quais colunas apareceram (nome do
município e valor do IDHM) — normalmente vem como `Município` e `IDHM 2010`
(o Atlas Brasil ainda não tem IDHM oficial pós-Censo 2022 publicado; vamos
usar o de 2010 como proxy e declarar isso como limitação do estudo). Ajusto
a célula de merge abaixo com o nome certo.


## 3.3 Montar a base consolidada


In [ ]:
# Base de internações: uma linha por (município, ano, causa) -> vamos pivotar
# para uma linha por (município, ano), com uma coluna por causa
internacoes_wide = (
    internacoes
    .pivot_table(index=["codigo_datasus", "ano"], columns="causa", values="internacoes", fill_value=0)
    .reset_index()
)
internacoes_wide["total_internacoes_causas_estudo"] = internacoes_wide[list(config.CAUSAS_CID10.keys())].sum(axis=1)

# Junta com a tabela de municípios para recuperar o código IBGE (7 dígitos)
base = internacoes_wide.merge(municipios, on="codigo_datasus", how="left")

# Junta com o censo (idosos sozinhos / total de idosos) -- constante entre os anos
# pois o Censo 2022 é um retrato único; ajuste se o nome da coluna do censo vier diferente
base = base.merge(censo, on="codigo_ibge", how="left")

# Junta com IDH, se disponível
# ATENÇÃO: ajustar left_on/right_on conforme as colunas reais do arquivo do Atlas Brasil
# if idh is not None:
#     base = base.merge(idh[["Município", "IDHM 2010"]], left_on="municipio", right_on="Município", how="left")

print(base.shape)
base.head()


In [ ]:
# Taxa de internação por 100 mil idosos, para poder comparar municípios de tamanhos diferentes
# ATENÇÃO: ajustar "total_idosos" para o nome real da coluna vinda do censo (seção 01)
# base["taxa_internacao_100k_idosos"] = base["total_internacoes_causas_estudo"] / base["total_idosos"] * 100_000
# base["pct_idosos_sozinhos"] = base["idosos_sozinhos"] / base["total_idosos"] * 100

print("Ajuste esta célula com os nomes reais das colunas do censo (notebook 01) antes de seguir.")


## 3.4 Destacar Rio Claro e salvar


In [ ]:
rio_claro = base[base["codigo_ibge"] == config.RIO_CLARO_CODIGO_IBGE]
print(f"Linhas de Rio Claro na base: {len(rio_claro)}")
rio_claro


In [ ]:
base.to_csv(config.DATA_PROCESSED / "dataset_consolidado_sp.csv", index=False)
print("Salvo em", config.DATA_PROCESSED / "dataset_consolidado_sp.csv")
